# Revisión de configuraciones atípicas · BGDY3x3 v3/v1

Localiza en `train.xyz` / `test.xyz` las configuraciones con energía anómalamente alta o geometrías
imposibles, las dibuja con sus fuerzas y permite exportar un dataset filtrado.

Para cada configuración se calcula:

| columna | significado |
|---|---|
| `E_int_at` | (E − Σ E0) / N, en eV/átomo (energía de interacción por átomo con los mismos E0s del `.yml`) |
| `dE_at` | `E_int_at` menos la mediana del train |
| `dmin` | distancia interatómica mínima (con PBC), en Å |
| `Fmax`, `Fmedia` | máximo y media de \|F\| por átomo, en eV/Å |
| `eps_xx`, `eps_yy`, `eps_xy`, `eps_area` | deformación de la celda respecto a la celda de equilibrio (la de la MD) |
| `dz` | espesor de la lámina (z máx − z mín), en Å; 0 = perfectamente plana |

`idx` es la posición (0-based) en el fichero, la misma que usa `ase.io.read(fichero, index=idx)`.

Ejecuta el notebook desde la carpeta que contiene `Dataset_BGDY3x3_v3/` (o cambia `DATA_DIR`).
Requiere `ase`, `numpy`, `pandas` y `matplotlib`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from ase.io import read, write
from ase.neighborlist import neighbor_list

# ---------------- Parametros (ajusta a tu caso) ----------------
DATA_DIR   = Path("dataset_xyz_v33")
TRAIN_FILE = DATA_DIR / "train_v33.xyz"
TEST_FILE  = DATA_DIR / "test_v33.xyz"
ENERGY_KEY = "energy_qe"
FORCES_KEY = "forces_qe"
E0S = {5: -149.49305965, 6: -242.95557274}   # los mismos E0s que en el .yml

# Criterios para marcar una configuracion como sospechosa
DE_MAX   = 1.0    # eV/atomo por encima de la mediana de E_int/atomo del train
DMIN_MIN = 1.00   # Å: distancia minima admisible (la MD a 2500 K no baja de ~1.09 Å)
FMAX_MAX = None   # eV/Å sobre |F| de un atomo; None = no se usa como criterio
# ----------------------------------------------------------------

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

## 1 · Descriptores por configuración

In [ ]:
def familia(ct: str) -> str:
    """'T2250K' -> 'MD', 'strain_uni-x' -> 'strain', 'rattle_0.04' -> 'rattle'."""
    if ct.startswith("T") and ct.endswith("K"):
        return "MD"
    return ct.split("_")[0]


def celda_referencia(frames):
    """Celda mas repetida entre los frames de MD (= celda de equilibrio)."""
    md = [a for a in frames if familia(a.info.get("config_type", "")) == "MD"] or frames
    claves = np.array([np.round(a.cell.array, 4).ravel() for a in md])
    vals, cuentas = np.unique(claves, axis=0, return_counts=True)
    return vals[cuentas.argmax()].reshape(3, 3)


def describir(frames, cell_ref):
    inv_ref = np.linalg.inv(cell_ref)
    filas = []
    for i, a in enumerate(frames):
        n = len(a)
        e_int = (a.info[ENERGY_KEY] - sum(E0S[z] for z in a.numbers)) / n
        fnorm = np.linalg.norm(a.arrays[FORCES_KEY], axis=1)
        d = neighbor_list("d", a, 3.0)
        D = inv_ref @ a.cell.array            # celda = celda_ref @ D
        eps = 0.5 * (D + D.T) - np.eye(3)     # deformacion (pequeñas deformaciones)
        z = a.positions[:, 2]
        ct = a.info.get("config_type", "-")
        filas.append(dict(
            idx=i, config_type=ct, familia=familia(ct),
            E_int_at=e_int,
            Fmax=fnorm.max(), Fmedia=fnorm.mean(),
            dmin=d.min() if len(d) else np.nan,
            dz=z.max() - z.min(),
            eps_xx=eps[0, 0], eps_yy=eps[1, 1], eps_xy=eps[0, 1],
            eps_area=abs(np.linalg.det(D[:2, :2])) - 1,
        ))
    return pd.DataFrame(filas).set_index("idx")


def marcar(df, e_ref):
    df["dE_at"] = df.E_int_at - e_ref
    df["por_E"] = df.dE_at > DE_MAX
    df["por_d"] = df.dmin < DMIN_MIN
    df["por_F"] = (df.Fmax > FMAX_MAX) if FMAX_MAX is not None else False
    df["sospechosa"] = df.por_E | df.por_d | df.por_F
    df["motivo"] = [
        ", ".join(m for m, f in zip(("energia", "dmin", "fuerza"), fila) if f)
        for fila in df[["por_E", "por_d", "por_F"]].itertuples(index=False)
    ]
    return df

In [ ]:
train = read(TRAIN_FILE, index=":")
test  = read(TEST_FILE,  index=":")

CELL_REF = celda_referencia(train)
df_train = describir(train, CELL_REF)
df_test  = describir(test,  CELL_REF)
E_REF = df_train.E_int_at.median()
df_train = marcar(df_train, E_REF)
df_test  = marcar(df_test,  E_REF)

print(f"train: {len(train)} configs | test: {len(test)} configs")
print(f"Mediana E_int (train): {E_REF:.3f} eV/at")
print("Celda de referencia (equilibrio, tomada de la MD):")
print(np.round(CELL_REF, 3))

md_ = df_train[df_train.familia == "MD"]
if len(md_):
    print(f"\nRango que cubre la MD del train ({', '.join(sorted(md_.config_type.unique()))}):")
    print(f"  E_int/at  {md_.E_int_at.min():.3f} … {md_.E_int_at.max():.3f} eV  (dE_at máx = {md_.dE_at.max():+.2f})")
    print(f"  dmin mín  {md_.dmin.min():.3f} Å   |F| máx {md_.Fmax.max():.1f} eV/Å   espesor dz máx {md_.dz.max():.2f} Å")

In [ ]:
def resumen(df):
    return (df.groupby("config_type")
              .agg(n=("E_int_at", "size"),
                   E_med=("E_int_at", "median"), E_max=("E_int_at", "max"),
                   dmin_min=("dmin", "min"), Fmax_max=("Fmax", "max"),
                   dz_max=("dz", "max"), sospechosas=("sospechosa", "sum"))
              .sort_values("E_max", ascending=False)
              .round(3))

print("Resumen por config_type (train)")
display(resumen(df_train))

## 2 · Configuraciones sospechosas

In [ ]:
COLS = ["config_type", "motivo", "E_int_at", "dE_at", "dmin", "Fmax",
        "eps_xx", "eps_yy", "eps_xy", "eps_area", "dz"]

sosp_train = df_train[df_train.sospechosa].sort_values("dE_at", ascending=False)
sosp_test  = df_test[df_test.sospechosa].sort_values("dE_at", ascending=False)

for nombre, df in (("train", df_train), ("test", df_test)):
    print(f"{nombre}: {df.sospechosa.sum()} sospechosas  "
          f"(solo energia: {(df.por_E & ~df.por_d).sum()}, solo dmin: {(df.por_d & ~df.por_E).sum()}, "
          f"ambas: {(df.por_E & df.por_d).sum()}, fuerza: {int(np.sum(df.por_F))})")

print("\nTRAIN")
display(sosp_train[COLS].round(3))
print("TEST")
display(sosp_test[COLS].round(3))

## 3 · Dónde caen frente al resto del dataset

In [ ]:
# Paleta categorica validada (orden fijo) + tintas neutras
COL  = {"MD": "#2a78d6", "rattle": "#eb6834", "strain": "#1baf7a"}
MK   = {"MD": "o", "rattle": "s", "strain": "^"}
ROJO = "#d03b3b"
TINTA, TINTA2, MUDO = "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9.5,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA2, "axes.titlecolor": TINTA,
    "xtick.color": MUDO, "ytick.color": MUDO,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})


def figura_resumen(df, nombre, n_etiquetas=8):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), layout="constrained")
    fams = [f for f in ("MD", "rattle", "strain") if f in set(df.familia)]

    # (a) Energia frente a distancia minima
    for f in fams:
        s = df[df.familia == f]
        ax1.scatter(s.dmin, s.E_int_at, s=16, marker=MK[f], color=COL[f], alpha=0.8,
                    linewidths=0, label=f"{f} (n={len(s)})")
    sos = df[df.sospechosa]
    ax1.scatter(sos.dmin, sos.E_int_at, s=90, facecolors="none", edgecolors=ROJO,
                linewidths=1.3, label=f"sospechosa (n={len(sos)})")
    ax1.axvline(DMIN_MIN, color=ROJO, ls="--", lw=1)
    ax1.axhline(E_REF + DE_MAX, color=ROJO, ls=":", lw=1)
    ax1.text(DMIN_MIN, ax1.get_ylim()[1], f" dmin = {DMIN_MIN:.2f} Å", color=TINTA2, va="top", fontsize=8)
    ax1.text(ax1.get_xlim()[1], E_REF + DE_MAX, f"mediana + {DE_MAX:g} eV/at ", color=TINTA2,
             ha="right", va="bottom", fontsize=8)
    for idx, r in sos.head(n_etiquetas).iterrows():
        ax1.annotate(f"#{idx}", (r.dmin, r.E_int_at), xytext=(5, 2), textcoords="offset points",
                     fontsize=7.5, color=TINTA2)
    ax1.set_xlabel("distancia interatómica mínima (Å)")
    ax1.set_ylabel("E_int (eV/átomo)")
    ax1.set_title(f"{nombre}: energía frente a distancia mínima", loc="left")
    ax1.legend(loc="lower right", fontsize=8)

    # (b) Distribucion de |F|max por familia
    bins = np.linspace(0, df.Fmax.max() * 1.02, 40)
    for f in fams:
        ax2.hist(df.Fmax[df.familia == f], bins=bins, histtype="step", lw=2, color=COL[f], label=f)
    if FMAX_MAX is not None:
        ax2.axvline(FMAX_MAX, color=ROJO, ls="--", lw=1)
    ax2.set_yscale("log")
    ax2.set_xlabel("|F| máxima en la configuración (eV/Å)")
    ax2.set_ylabel("nº configuraciones")
    ax2.set_title(f"{nombre}: fuerzas máximas por familia", loc="left")
    ax2.legend(fontsize=8)
    plt.show()


figura_resumen(df_train, "train")
figura_resumen(df_test, "test")

### Curvas de energía de las configuraciones deformadas

Las `strain_*` deberían caer en curvas suaves E(ε). La banda gris es el rango de energías que cubre la MD:
lo que queda muy por encima son estados que tu MD no visita.

In [ ]:
def curvas_strain(df, nombre):
    s = df[df.familia == "strain"]
    if s.empty:
        print(f"{nombre}: no hay configuraciones strain"); return
    tipos = sorted(s.config_type.unique())
    paleta = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
    marcas = ["o", "s", "^", "D", "v", "P", "X", "*"]
    fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")
    md_ = df_train[df_train.familia == "MD"]
    if len(md_):
        ax.axhspan(md_.E_int_at.min(), md_.E_int_at.max(), color="#c3c2b7", alpha=0.35, lw=0,
                   label="rango MD (train)")
    for k, t in enumerate(tipos):
        g = s[s.config_type == t].sort_values("eps_area")
        ls = "none" if "aniso" in t else "-"
        ax.plot(g.eps_area * 100, g.E_int_at, ls=ls, lw=2, marker=marcas[k], ms=5,
                color=paleta[k], label=f"{t} (n={len(g)})")
    sos = s[s.sospechosa]
    ax.scatter(sos.eps_area * 100, sos.E_int_at, s=110, facecolors="none", edgecolors=ROJO,
               linewidths=1.3, zorder=5, label="sospechosa")
    for idx, r in sos.iterrows():
        ax.annotate(f"#{idx}", (r.eps_area * 100, r.E_int_at), xytext=(6, 0),
                    textcoords="offset points", fontsize=7.5, color=TINTA2, va="center")
    ax.set_xlabel("deformación de área  ε_A = A/A₀ − 1  (%)")
    ax.set_ylabel("E_int (eV/átomo)")
    ax.set_title(f"{nombre}: energía de las configuraciones deformadas", loc="left")
    ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))
    plt.show()


curvas_strain(df_train, "train")

## 4 · Cómo son las estructuras

Vista superior (con enlaces < 1.9 Å y fuerzas en el plano) y vista lateral (x–z, con fuerzas x–z).
El enlace más corto se marca en rojo. Todas las columnas comparten la escala de flechas y de color,
así que las fuerzas se comparan directamente. La primera columna es una referencia casi sin deformar.

In [ ]:
COL_EL = {6: "#52514e", 5: "#eb6834"}          # C gris oscuro, B naranja
CMAP_F = LinearSegmentedColormap.from_list("fuerza", ["#86b6ef", "#2a78d6", "#184f95", "#0d366b"])


def dibujar(atoms, ax_sup, ax_lat, titulo, f_escala):
    a = atoms.copy()
    a.wrap()
    pos = a.positions
    f = a.arrays[FORCES_KEY]
    fn = np.linalg.norm(f, axis=1)
    norm = Normalize(0, f_escala)
    largo = f_escala / 3.0                     # |F| = f_escala -> flecha de 3 Å
    colores = [COL_EL.get(z, MUDO) for z in a.numbers]

    i, j, D = neighbor_list("ijD", a, 1.9)
    m = i < j
    for ax, (c0, c1) in ((ax_sup, (0, 1)), (ax_lat, (0, 2))):
        segs = np.stack([pos[i[m]][:, [c0, c1]], pos[i[m]][:, [c0, c1]] + D[m][:, [c0, c1]]], axis=1)
        ax.add_collection(LineCollection(segs, colors="#c3c2b7", linewidths=0.9, zorder=1))
        if len(D):
            k = np.argmin(np.linalg.norm(D, axis=1))
            p0 = pos[i[k]][[c0, c1]]
            p1 = p0 + D[k][[c0, c1]]
            ax.plot(*np.stack([p0, p1]).T, color=ROJO, lw=3, zorder=4, solid_capstyle="round")
        ax.scatter(pos[:, c0], pos[:, c1], s=18, c=colores, edgecolors="white", linewidths=0.4, zorder=2)
        ax.quiver(pos[:, c0], pos[:, c1], f[:, c0], f[:, c1], fn, cmap=CMAP_F, norm=norm,
                  angles="xy", scale_units="xy", scale=largo, width=0.0035, zorder=3)
        ax.grid(False)
        ax.set_aspect("equal")

    c = a.cell.array
    caja = np.array([[0, 0], c[0, :2], c[0, :2] + c[1, :2], c[1, :2], [0, 0]])
    ax_sup.plot(caja[:, 0], caja[:, 1], color=MUDO, lw=0.8, ls="--", zorder=0)
    ax_sup.set_title(titulo, fontsize=8.5, loc="left")
    ax_sup.set_xlabel("x (Å)"); ax_sup.set_ylabel("y (Å)")
    zc = 0.5 * (pos[:, 2].max() + pos[:, 2].min())
    ax_lat.set_ylim(zc - 5, zc + 5)
    ax_lat.set_xlabel("x (Å)"); ax_lat.set_ylabel("z (Å)")


def comparar(frames, df, indices, etiqueta, nombres=None):
    n = len(indices)
    fig, axs = plt.subplots(2, n, figsize=(4.6 * n, 4.9), squeeze=False, layout="constrained",
                            gridspec_kw={"height_ratios": [3, 1]})
    f_escala = float(df.loc[indices, "Fmax"].max())
    for col, idx in enumerate(indices):
        r = df.loc[idx]
        cab = nombres[col] if nombres else f"{etiqueta} #{idx}"
        tit = (f"{cab} · {r.config_type}\n"
               f"ΔE = {r.dE_at:+.2f} eV/at · dmin = {r.dmin:.3f} Å\n"
               f"|F|max = {r.Fmax:.1f} eV/Å · ε_xx = {100*r.eps_xx:+.1f}% · ε_yy = {100*r.eps_yy:+.1f}%")
        dibujar(frames[idx], axs[0, col], axs[1, col], tit, f_escala)
    sm = plt.cm.ScalarMappable(norm=Normalize(0, f_escala), cmap=CMAP_F)
    fig.colorbar(sm, ax=axs[0, :].tolist(), shrink=0.6, label="|F| (eV/Å)")
    plt.show()


# Referencia: la configuracion strain menos deformada (o la MD mas cercana a la mediana)
cand = df_train[df_train.familia == "strain"]
REF_IDX = int((cand.eps_xx.abs() + cand.eps_yy.abs() + cand.eps_xy.abs()).idxmin()) if len(cand) \
          else int(df_train.dE_at.abs().idxmin())
print("Referencia:", REF_IDX, df_train.loc[REF_IDX, "config_type"])

In [ ]:
# Las 3 peores del train (por energia) frente a la referencia.
# Cambia VER por los indices que quieras inspeccionar, p. ej. VER = [18, 36, 76].
VER = list(sosp_train.index[:3])
comparar(train, df_train, [REF_IDX] + VER, "train")

In [ ]:
# Sospechosas del test (frente a la misma referencia del train)
if len(sosp_test):
    n = min(3, len(sosp_test))
    fig_frames = [train[REF_IDX]] + [test[i] for i in sosp_test.index[:n]]
    fig_df = pd.concat([df_train.loc[[REF_IDX]], df_test.loc[sosp_test.index[:n]]]).reset_index(drop=True)
    nombres = [f"train #{REF_IDX} (ref)"] + [f"test #{i}" for i in sosp_test.index[:n]]
    comparar(fig_frames, fig_df, list(range(n + 1)), "test", nombres=nombres)
else:
    print("No hay sospechosas en el test")

In [ ]:
# Visor 3D interactivo de una configuracion concreta (nglview si esta instalado; si no, x3d)
from ase.visualize import view

IDX = int(sosp_train.index[0]) if len(sosp_train) else REF_IDX
try:
    import nglview  # noqa: F401
    visor = view(train[IDX], viewer="ngl")
except ImportError:
    visor = view(train[IDX], viewer="x3d")
visor

## 5 · Exportar

Siempre se guardan las configuraciones descartadas en `revision_outliers/`, para abrirlas en OVITO, VESTA o ASE.
Los datasets filtrados solo se escriben si pones `EXPORTAR = True`.

In [ ]:
# --- Elige que descartar -------------------------------------------------------
#   df.sospechosa          -> cualquier criterio (energia, dmin o fuerza)
#   df.por_d               -> solo geometrias con distancias imposibles
#   df.por_E & df.por_d    -> las que cumplen ambos
DESCARTAR_TRAIN = set(df_train.index[df_train.sospechosa])
DESCARTAR_TEST  = set(df_test.index[df_test.sospechosa])

# Ajustes manuales (indices tal y como salen en las tablas)
DESCARTAR_TRAIN |= set([])     # añadir
DESCARTAR_TRAIN -= set([])     # rescatar
DESCARTAR_TEST  |= set([])
DESCARTAR_TEST  -= set([])

EXPORTAR = True
OUT_DIR  = DATA_DIR.parent / f"{DATA_DIR.name}_filtrado"
REV_DIR  = Path("revision_outliers")
# --------------------------------------------------------------------------------

REV_DIR.mkdir(exist_ok=True)
for nombre, frames, drop in (("train", train, DESCARTAR_TRAIN), ("test", test, DESCARTAR_TEST)):
    sel = [frames[i] for i in sorted(drop)]
    if sel:
        write(REV_DIR / f"descartadas_{nombre}.xyz", sel, format="extxyz")
    print(f"{nombre}: descartar {len(drop)} de {len(frames)} -> quedan {len(frames) - len(drop)}"
          f"   índices: {sorted(int(i) for i in drop)}")

if EXPORTAR:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for nombre, frames, drop in (("train", train, DESCARTAR_TRAIN), ("test", test, DESCARTAR_TEST)):
        keep = [a for i, a in enumerate(frames) if i not in drop]
        out = OUT_DIR / f"{nombre}.xyz"
        write(out, keep, format="extxyz")
        # Comprobacion: se relee y se verifica que no se han perdido energias ni fuerzas
        back = read(out, index=":")
        assert len(back) == len(keep)
        assert all(ENERGY_KEY in b.info and FORCES_KEY in b.arrays for b in back)
        assert np.allclose([b.info[ENERGY_KEY] for b in back], [a.info[ENERGY_KEY] for a in keep])
        print(f"Escrito y verificado: {out} ({len(back)} configs)")
    print("\nCambia en el .yml (y usa otro name/model_dir para que restart_latest no retome el FT anterior):")
    print(f'train_file: "{OUT_DIR / "train.xyz"}"')
    print(f'test_file: "{OUT_DIR / "test.xyz"}"')
else:
    print(f"\nEXPORTAR = False: no se han escrito datasets filtrados (se escribirían en {OUT_DIR}/)")